In [2]:
import os
from typing import Annotated, List, TypedDict, Optional, Union
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\ASUS\OneDrive\Desktop\DeepForge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# The specialized framework for deep reasoning agents
from deepagents import create_deep_agent

In [4]:
load_dotenv()

True

In [6]:
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    paper_path: Optional[str]
    scientific_logic: Optional[str]
    implementation_code: Optional[str]

print("Environment and State initialized.")

Environment and State initialized.


In [7]:
llm= ChatGoogleGenerativeAI(
    model= "gemini-2.0-flash",
    temperature= 0
)

In [8]:
research_agent= create_deep_agent(
    model= llm,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your job is to read research papers, extract the mathematical core, "
        "and explain the architecture in plain pseudocode. "
        "Use your planning tools to break down the paper analysis step-by-step."
    )
)
print("Deep Research Agent created.")

Deep Research Agent created.


In [11]:
import arxiv
import fitz  
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

In [13]:
# 1. arXiv Search and Download Tool
@tool
def download_research_paper(query: str)-> str:
    """
    Searches arXiv for a paper, downloads the PDF to the '../data/' folder, 
    and returns the local path to the file.
    """
    download_path= "../data/"
    os.makedirs(download_path, exist_ok= True)

    search= arxiv.Search(query= query, max_results= 1, sort_by= aarxiv.SortCriterion.Relevance)
    paper= next(search.results())

    filename = f"{paper.title.replace(' ', '_').replace(':', '')}.pdf"
    full_path = paper.download_pdf(dirpath=download_path, filename=filename)
    
    return f"Paper downloaded to: {full_path}"

In [14]:
# 2. PDF Reading Tool
@tool
def read_pdf_content(file_path: str) -> str:
    """
    Opens a PDF file and extracts the text content. 
    Useful for reading downloaded research papers.
    """
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [15]:
# 3. Web Search Tool (Tavily)
web_search = TavilySearchResults(k=3)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_2336\3970997022.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(k=3)


In [16]:
# Combine all tools
tools = [download_research_paper, read_pdf_content, web_search]
print("ArXiv, PDF, and Tavily tools defined.")

ArXiv, PDF, and Tavily tools defined.
